# XGBoost Pipeline — Trajectory Prediction Metrics (GAM-Style)

This notebook fits an XGBoost regressor to trajectory prediction error metrics using prepared trajectory/scene characteristics.
It mirrors the structure of `gam.ipynb`, uses a random train/validation/test split, and focuses interpretability on conditions where performance is poor vs. well.

## 1. Imports and Configuration

In [ ]:
# Core libraries
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import xgboost as xgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Notebook-level config constants
MODEL_ID = 'xgboost'
RUN_NAME = "nusc_mini_debug_tpp-11_Mar_2026_15_29_02"
PREPARED_TARGET_COL = "ml_ade"
DATA_PATH = Path('../../results/interpretable_model/prepared_data') / RUN_NAME / f"prepared_data_{PREPARED_TARGET_COL}.csv"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'

RANDOM_STATE = 42
k_outer_fold = 5
k_inner_fold = 3
N_OPTUNA_TRIALS = 40
MAX_BOOST_ROUNDS = 2000
EARLY_STOPPING_ROUNDS = 50
N_JOBS = -1
POOR_WELL_QUANTILE = 0.20
TUNING_METRIC = 'rmse'

SAVE_DIR = Path('../../results/interpretable_model') / MODEL_ID / RUN_NAME
PLOTS_DIR = SAVE_DIR / 'plots'
TABLES_DIR = SAVE_DIR / 'tables'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports and configuration loaded.')
print(f'Run: {RUN_NAME}')
print(f'Model ID: {MODEL_ID}')
print('Interpretable model: XGBoost')
print(f'DATA_PATH: {DATA_PATH}')
print(f'SAVE_DIR:  {SAVE_DIR.resolve()}')
print(
    'Nested CV: '
    f'outer={k_outer_fold}, inner={k_inner_fold}, optuna_trials={N_OPTUNA_TRIALS}, '
    f'early_stop={EARLY_STOPPING_ROUNDS}'
)


## 2. Load Prepared Data and Inspect

In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print('Columns:')
print(df.columns.tolist())

display(df.head())

missing_summary = df.isna().sum().sort_values(ascending=False)
print('\nMissing values per column:')
display(missing_summary.to_frame('missing_count'))

dtype_summary = df.dtypes.astype(str).to_frame('dtype')
print('\nColumn dtypes:')
display(dtype_summary)

Dataset shape: (451, 14)
Columns:
['max_speed', 'std_speed', 'mean_acceleration', 'max_acceleration', 'path_efficiency', 'heading_change', 'has_collision', 'min_neighbor_distance', 'scene_num_agents', 'scene_bbox_area', 'scene_bbox_width', 'scene_spatial_density', 'scene_density_VEHICLE', 'ml_ade_log']


,max_speed,std_speed,mean_acceleration,max_acceleration,path_efficiency,heading_change,has_collision,min_neighbor_distance,scene_num_agents,scene_bbox_area,scene_bbox_width,scene_spatial_density,scene_density_VEHICLE,ml_ade_log
0,1.230403,0.116722,0.056417,0.404461,84.461237,112.440745,0.0,1.013903,17.0,1160.377375,40.869190,0.014650,0.005171,0.649643
1,0.218477,0.063105,0.007725,0.311925,15.733080,941.304051,0.0,3.023524,17.0,1160.377375,40.869190,0.014650,0.005171,0.140704
2,0.383437,0.099171,0.031384,0.470005,43.684220,488.173829,0.0,1.104355,50.0,1821.083675,44.533135,0.027456,0.012630,0.388765
3,1.483624,0.190256,0.051802,0.794096,95.139840,81.155053,1.0,0.426945,16.0,1230.401053,43.391608,0.013004,0.005689,0.821990
4,0.218477,0.064457,0.010995,0.311925,27.282729,946.683531,0.0,2.796330,15.0,1057.088774,37.340273,0.014190,0.005676,0.159340



Missing values per column:


,missing_count
max_speed,0
std_speed,0
mean_acceleration,0
max_acceleration,0
path_efficiency,0
heading_change,0
has_collision,0
min_neighbor_distance,0
scene_num_agents,0
scene_bbox_area,0



Column dtypes:


,dtype
max_speed,float64
std_speed,float64
mean_acceleration,float64
max_acceleration,float64
path_efficiency,float64
heading_change,float64
has_collision,float64
min_neighbor_distance,float64
scene_num_agents,float64
scene_bbox_area,float64


## 3. Resolve Target and Build Feature Matrix

In [3]:
if TARGET_COL is not None:
    assert TARGET_COL in df.columns, f'TARGET_COL={TARGET_COL} not found in dataset columns.'
    target_col = TARGET_COL
else:
    if 'ml_ade_log' in df.columns:
        target_col = 'ml_ade_log'
    elif 'ml_ade' in df.columns:
        target_col = 'ml_ade'
    else:
        target_col = df.columns[-1]

feature_cols = [c for c in df.columns if c != target_col]

# Keep only numeric features (prepared dataset is expected to be numeric-only)
non_numeric_features = [c for c in feature_cols if not np.issubdtype(df[c].dtype, np.number)]
if non_numeric_features:
    print('WARNING: Non-numeric features found and dropped:')
    print(non_numeric_features)
feature_cols = [c for c in feature_cols if c not in non_numeric_features]

model_df = df[feature_cols + [target_col]].dropna().copy()

X = model_df[feature_cols]
y = model_df[target_col]
row_ids = model_df.index.to_numpy()

print(f'Target column: {target_col}')
print(f'Number of features: {len(feature_cols)}')
print(f'Rows available for modeling: {len(model_df)}')
print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')

Target column: ml_ade_log
Number of features: 13
Rows available for modeling: 451
Feature matrix shape: (451, 13)
Target vector shape: (451,)


## 4. Outer/Inner CV Setup (OOF + Optuna)


In [4]:
def to_original_scale(values, col_name):
    values = np.asarray(values)
    if col_name.endswith('_log'):
        return np.expm1(values)
    return values

outer_cv = KFold(n_splits=k_outer_fold, shuffle=True, random_state=RANDOM_STATE)

n_samples = len(model_df)
oof_pred = np.full(n_samples, np.nan, dtype=float)
oof_fold = np.full(n_samples, -1, dtype=int)

print(f'Outer CV folds: {k_outer_fold}')
print(f'Inner CV folds: {k_inner_fold}')
print(f'Rows for OOF prediction: {n_samples}')


Outer CV folds: 5
Inner CV folds: 3
Rows for OOF prediction: 451


## 5. Nested Resampling + Optuna Hyperparameter Optimization (xgb.cv)


In [ ]:
base_params = {
    'objective': 'reg:squarederror',
    'eval_metric': TUNING_METRIC,
    'tree_method': 'hist',
    'verbosity': 0,
    'nthread': N_JOBS,
}


def run_optuna_tuning(X_train, y_train, *, seed, tuning_scope):
    dtrain = xgb.DMatrix(X_train, label=y_train)
    trial_rows = []

    def objective(trial):
        params = base_params.copy()
        params.update({
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 20.0, log=True),
            'seed': seed,
        })

        cv_results = xgb.cv(
            params=params,
            dtrain=dtrain,
            nfold=k_inner_fold,
            num_boost_round=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            metrics=TUNING_METRIC,
            seed=seed,
            shuffle=True,
            verbose_eval=False,
        )

        best_iteration = int(cv_results[f'test-{TUNING_METRIC}-mean'].idxmin() + 1)
        best_cv_score = float(cv_results[f'test-{TUNING_METRIC}-mean'].min())
        trial.set_user_attr('best_iteration', best_iteration)
        trial.set_user_attr('best_cv_score', best_cv_score)

        row = {
            'tuning_scope': tuning_scope,
            'trial_number': trial.number,
            'best_iteration': best_iteration,
            'best_cv_score': best_cv_score,
        }
        row.update({k: v for k, v in params.items() if k not in base_params})
        trial_rows.append(row)
        return best_cv_score

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

    trial_results_df = pd.DataFrame(trial_rows).sort_values('best_cv_score').reset_index(drop=True)
    best_iteration = int(study.best_trial.user_attrs['best_iteration'])
    best_cv_score = float(study.best_trial.user_attrs['best_cv_score'])

    print(
        f'{tuning_scope} tuning complete | '
        f'best_{TUNING_METRIC}={best_cv_score:.6f} | '
        f'best_iteration={best_iteration}'
    )

    return {
        'best_params': study.best_params,
        'best_iteration': best_iteration,
        'best_cv_score': best_cv_score,
        'trial_results_df': trial_results_df,
    }


nested_rows = []

for fold_idx, (outer_train_idx, outer_valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y.iloc[outer_train_idx]
    X_outer_valid = X.iloc[outer_valid_idx]
    y_outer_valid = y.iloc[outer_valid_idx]

    tuning_result = run_optuna_tuning(
        X_outer_train,
        y_outer_train,
        seed=RANDOM_STATE + fold_idx,
        tuning_scope=f'outer_fold_{fold_idx}',
    )

    train_params = base_params.copy()
    train_params.update(tuning_result['best_params'])
    train_params['seed'] = RANDOM_STATE + fold_idx

    dtrain = xgb.DMatrix(X_outer_train, label=y_outer_train)
    dvalid = xgb.DMatrix(X_outer_valid, label=y_outer_valid)
    booster = xgb.train(train_params, dtrain, num_boost_round=tuning_result['best_iteration'])
    oof_pred[outer_valid_idx] = booster.predict(dvalid)
    oof_fold[outer_valid_idx] = fold_idx

    y_outer_valid_orig = to_original_scale(y_outer_valid, target_col)
    outer_pred_orig = to_original_scale(oof_pred[outer_valid_idx], target_col)

    outer_rmse = float(np.sqrt(mean_squared_error(y_outer_valid_orig, outer_pred_orig)))
    outer_mae = mean_absolute_error(y_outer_valid_orig, outer_pred_orig)
    outer_r2 = r2_score(y_outer_valid_orig, outer_pred_orig)

    row = {
        'outer_fold': fold_idx,
        'outer_rmse': outer_rmse,
        'outer_mae': outer_mae,
        'outer_r2': outer_r2,
        'inner_best_cv_rmse': tuning_result['best_cv_score'],
        'best_iteration': tuning_result['best_iteration'],
    }
    row.update({f'best_{k}': v for k, v in tuning_result['best_params'].items()})
    nested_rows.append(row)

    print(
        f'Outer fold {fold_idx}/{k_outer_fold} | '
        f'RMSE={outer_rmse:.6f} | MAE={outer_mae:.6f} | R2={outer_r2:.4f}'
    )

if np.isnan(oof_pred).any():
    raise ValueError('OOF predictions contain NaN values. Check CV splits and training flow.')

nested_cv_df = pd.DataFrame(nested_rows)
display(nested_cv_df)

nested_summary = pd.DataFrame([
    {
        'metric': 'outer_rmse',
        'mean': nested_cv_df['outer_rmse'].mean(),
        'std': nested_cv_df['outer_rmse'].std(ddof=1),
    },
    {
        'metric': 'outer_mae',
        'mean': nested_cv_df['outer_mae'].mean(),
        'std': nested_cv_df['outer_mae'].std(ddof=1),
    },
    {
        'metric': 'outer_r2',
        'mean': nested_cv_df['outer_r2'].mean(),
        'std': nested_cv_df['outer_r2'].std(ddof=1),
    },
])
print('Nested CV summary:')
display(nested_summary)

nested_path = TABLES_DIR / f'nested_cv_optuna_{target_col}.csv'
nested_summary_path = TABLES_DIR / f'nested_cv_optuna_summary_{target_col}.csv'
nested_cv_df.to_csv(nested_path, index=False)
nested_summary.to_csv(nested_summary_path, index=False)
print(f'Nested CV fold results saved to: {nested_path}')
print(f'Nested CV summary saved to:      {nested_summary_path}')

# Build OOF prediction table
model_df_oof = model_df.copy()
model_df_oof['row_id'] = row_ids
model_df_oof['oof_pred'] = oof_pred
model_df_oof['outer_fold'] = oof_fold
model_df_oof['oof_pred_orig'] = to_original_scale(oof_pred, target_col)
model_df_oof['target_orig'] = to_original_scale(model_df_oof[target_col].values, target_col)

oof_path = TABLES_DIR / f'oof_predictions_{target_col}.csv'
model_df_oof[['row_id', 'oof_pred', 'oof_pred_orig', 'target_orig', 'outer_fold']].to_csv(oof_path, index=False)
print(f'OOF predictions saved to: {oof_path}')

full_data_tuning = run_optuna_tuning(
    X,
    y,
    seed=RANDOM_STATE,
    tuning_scope='full_data',
)
full_data_tuning_trials_path = TABLES_DIR / f'full_data_tuning_optuna_trials_{target_col}.csv'
full_data_tuning['trial_results_df'].to_csv(full_data_tuning_trials_path, index=False)

full_data_tuning_summary = {
    'model_id': MODEL_ID,
    'run_name': RUN_NAME,
    'target_col': target_col,
    'tuning_metric_name': TUNING_METRIC,
    'best_params': full_data_tuning['best_params'],
    'best_iteration': full_data_tuning['best_iteration'],
    'best_cv_score': full_data_tuning['best_cv_score'],
    'tuning_config': {
        'nfold': k_inner_fold,
        'n_trials': N_OPTUNA_TRIALS,
        'max_boost_rounds': MAX_BOOST_ROUNDS,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
        'random_state': RANDOM_STATE,
    },
}
full_data_tuning_summary_path = TABLES_DIR / f'full_data_tuning_optuna_summary_{target_col}.json'
full_data_tuning_summary_path.write_text(json.dumps(full_data_tuning_summary, indent=2))

print('Selected hyperparameters from full-data retuning:')
print(full_data_tuning['best_params'])
print(f"Best iteration from full-data retuning: {full_data_tuning['best_iteration']}")
print(f'Full-data tuning trials saved to: {full_data_tuning_trials_path}')
print(f'Full-data tuning summary saved to: {full_data_tuning_summary_path}')

model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=full_data_tuning['best_iteration'],
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbosity=0,
    **full_data_tuning['best_params'],
)
model.fit(X, y, verbose=False)
print('Final model fitted on full data using retuned hyperparameters.')


## 6. Evaluate on OOF Predictions (Original Scale)


In [6]:
def evaluate_oof(y_true_raw, y_pred_raw, col_name):
    y_true_orig = to_original_scale(np.asarray(y_true_raw), col_name)
    y_pred_orig = to_original_scale(np.asarray(y_pred_raw), col_name)

    return {
        'Split': 'OOF',
        'R²': r2_score(y_true_orig, y_pred_orig),
        'MAE': mean_absolute_error(y_true_orig, y_pred_orig),
        'RMSE': float(np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))),
    }, y_true_orig, y_pred_orig

oof_metrics, y_oof_orig, y_oof_pred_orig = evaluate_oof(y, oof_pred, target_col)

metrics_df = pd.DataFrame([oof_metrics])
display(metrics_df)

metrics_path = TABLES_DIR / f'metrics_oof_{target_col}.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'OOF metrics saved to: {metrics_path}')


,Split,R²,MAE,RMSE
0,OOF,0.779425,0.183176,0.273296


OOF metrics saved to: ../../results/interpretable_model/xgboost/tables/metrics_oof_ml_ade_log.csv


## 7. Export Artifacts and Run Manifest


In [ ]:
# Persist model for downstream analysis notebooks
model_path = SAVE_DIR / f'xgb_model_{target_col}.json'
model.save_model(model_path)

# Persist modeling data with OOF predictions
data_path = TABLES_DIR / f'model_data_with_oof_{target_col}.csv'
model_df_oof.to_csv(data_path, index=False)

manifest = {
    'model_id': MODEL_ID,
    'run_name': RUN_NAME,
    'target_col': target_col,
    'feature_cols': feature_cols,
    'artifact_root': str(SAVE_DIR),
    'plots_dir': str(PLOTS_DIR),
    'tables_dir': str(TABLES_DIR),
    'nested_resampling': {
        'nested_cv_path': str(nested_path),
        'nested_cv_summary_path': str(nested_summary_path),
        'oof_predictions_path': str(oof_path),
        'oof_metrics_path': str(metrics_path),
        'model_data_with_oof_path': str(data_path),
    },
    'final_model': {
        'model_path': str(model_path),
        'full_data_tuning_trials_path': str(full_data_tuning_trials_path),
        'full_data_tuning_summary_path': str(full_data_tuning_summary_path),
        'selected_best_params': full_data_tuning['best_params'],
        'selected_best_iteration': full_data_tuning['best_iteration'],
        'best_cv_score': full_data_tuning['best_cv_score'],
        'tuning_metric_name': TUNING_METRIC,
    },
    'analysis': {
        'poor_well_quantile': POOR_WELL_QUANTILE,
    },
}
manifest_path = TABLES_DIR / f'run_manifest_{target_col}.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f'Interpretable model saved to: {model_path}')
print(f'Model data saved to: {data_path}')
print(f'Run manifest saved to: {manifest_path}')
